# Being John Snow: Point Pattern Analysis & Spatial Statistics in Python

This Colab-ready notebook is a Python version of the classic **Being John Snow** lab. It combines the bullets from the short Week 08 notebook outline with the fuller QGIS-based archive version from the old course.

**Main ideas in this notebook:**

1. Voronoi mapping of water pumps and spatial join of deaths
2. Spatial mean, weighted spatial mean, standard ellipse, and standard distance
3. Network-based service areas that are similar to Voronoi polygons, but use travel time instead of straight-line distance
4. Kernel density surfaces to highlight hotspots

> The data archive for this lab is `Being_John_Snow.zip`, which lives in the course `data/` directory. If you run this notebook in Colab, it will download the archive from GitHub automatically.

## Install the Python packages

Run this cell first in Colab. It installs the geospatial packages used below.

In [ ]:
!pip -q install geopandas shapely pyogrio rasterio contextily mapclassify

## Optional install for the network-service-area extension

If you want to explore the network-based extension at the end of the notebook, install `osmnx` too.

In [ ]:
!pip -q install osmnx

## Import libraries and download the data archive

This notebook expects to run from Colab or from a local Python environment that has internet access. It downloads the John Snow archive if it is not already present, then unpacks it.

In [ ]:
from pathlib import Path
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import geopandas as gpd
from shapely.geometry import Point, Polygon
from scipy.spatial import Voronoi
from scipy.stats import gaussian_kde
from pyproj import CRS

plt.style.use('seaborn-v0_8-whitegrid')

DATA_URL = 'https://github.com/mapninja/Earthsys144/raw/master/data/Being_John_Snow.zip'
ZIP_PATH = Path('Being_John_Snow.zip')
EXTRACT_DIR = Path('Being_John_Snow')

if not ZIP_PATH.exists():
    print('Downloading John Snow data archive...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not EXTRACT_DIR.exists():
    print('Unpacking archive...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')

RAW_DIR = EXTRACT_DIR / 'data' / 'raw'
MAP_PATH = RAW_DIR / 'John_Snow_Map.tif'
PUMP_PATH = RAW_DIR / 'Water_Pumps.geojson'
DEATH_PATH = RAW_DIR / 'deathAddresses.csv'
STUDY_AREA_PATH = RAW_DIR / 'Study_Area.shp'

print('Data folder:', RAW_DIR.resolve())
print('Files found:', PUMP_PATH.exists(), DEATH_PATH.exists(), STUDY_AREA_PATH.exists(), MAP_PATH.exists())

## Load the points, pumps, and study area

The death addresses are stored in a CSV, so we convert the longitude/latitude columns into a point layer. We will then project everything to a metric CRS for distance-based analysis.

In [ ]:
deaths_df = pd.read_csv(DEATH_PATH)
print('Death table columns:', deaths_df.columns.tolist())
print(deaths_df.head())

deaths = gpd.GeoDataFrame(
    deaths_df,
    geometry=gpd.points_from_xy(deaths_df['xcoord'], deaths_df['ycoord']),
    crs='EPSG:4326'
)

pumps = gpd.read_file(PUMP_PATH)
study_area = gpd.read_file(STUDY_AREA_PATH)

# A projected CRS is important for distances, density surfaces, and ellipse calculations.
# British National Grid is a good choice for central London.
analysis_crs = 'EPSG:27700'

deaths_proj = deaths.to_crs(analysis_crs)
pumps_proj = pumps.to_crs(analysis_crs)
study_proj = study_area.to_crs(analysis_crs)

print('Pumps columns:', pumps.columns.tolist())
print('Study area CRS:', study_area.crs)
print('Projected CRS:', analysis_crs)

## Quick map of the raw data

This shows the deaths, pumps, and study area before we do any analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
study_proj.boundary.plot(ax=ax, color='black', linewidth=1.5)
pumps_proj.plot(ax=ax, color='royalblue', markersize=40, label='Water pumps')
deaths_proj.plot(ax=ax, color='crimson', markersize=np.clip(deaths_proj['Num_Cases'] * 12, 12, 120), alpha=0.6, label='Deaths')
ax.set_title('John Snow study area: pumps and death addresses')
ax.set_axis_off()
ax.legend(loc='lower left')
plt.show()

## Voronoi polygons for pump service areas

Thiessen/Voronoi polygons allocate space to the nearest pump. In the notebook, we build the polygons from the pump coordinates and clip them to the study area.

In [ ]:
def voronoi_finite_polygons_2d(vor, radius=None):
    """Reconstruct infinite Voronoi regions in a 2D diagram to finite regions.

    Adapted from the SciPy cookbook / StackOverflow recipe used widely for plotting.
    Returns regions and vertices as index lists.
    """
    if vor.points.shape[1] != 2:
        raise ValueError('Requires 2D input')

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max() * 2

    # Construct a map containing all ridges for a given point
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]
        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue

        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0 and v2 >= 0:
                continue

            # Compute the missing endpoint of an infinite ridge
            t = vor.points[p2] - vor.points[p1]
            t = t / np.linalg.norm(t)
            n = np.array([-t[1], t[0]])

            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius
            new_vertices.append(far_point.tolist())
            new_region.append(len(new_vertices) - 1)

        # Sort region vertices in clockwise order
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:, 1] - c[1], vs[:, 0] - c[0])
        new_region = np.array(new_region)[np.argsort(angles)].tolist()
        new_regions.append(new_region)

    return new_regions, np.asarray(new_vertices)

pump_xy = np.array([(geom.x, geom.y) for geom in pumps_proj.geometry])
vor = Voronoi(pump_xy)
regions, vertices = voronoi_finite_polygons_2d(vor)
study_union = study_proj.geometry.unary_union

polygon_geoms = []
for region in regions:
    polygon = Polygon(vertices[region])
    polygon = polygon.intersection(study_union)
    polygon_geoms.append(polygon)

voronoi_gdf = gpd.GeoDataFrame(
    pumps_proj[['Name']].copy(),
    geometry=polygon_geoms,
    crs=analysis_crs
)

fig, ax = plt.subplots(figsize=(10, 10))
voronoi_gdf.boundary.plot(ax=ax, color='darkorange', linewidth=1)
pumps_proj.plot(ax=ax, color='royalblue', markersize=40)
study_proj.boundary.plot(ax=ax, color='black', linewidth=1.5)
ax.set_title('Voronoi pump service areas')
ax.set_axis_off()
plt.show()

## Spatial join: assign each death to its nearest pump

Once the Voronoi polygons are built, every death point can be tagged with the pump whose polygon contains it.

In [ ]:
death_joined = gpd.sjoin(
    deaths_proj,
    voronoi_gdf[['Name', 'geometry']],
    how='left',
    predicate='within'
)

summary = (
    death_joined
    .groupby('Name', dropna=False)
    .agg(total_deaths=('Num_Cases', 'sum'),
         death_addresses=('Num_Cases', 'count'),
         mean_cases=('Num_Cases', 'mean'))
    .sort_values('total_deaths', ascending=False)
)

print(summary)

## Spatial mean, weighted spatial mean, and standard ellipse

The old QGIS lab used mean centers to show how the deaths cluster around Broad Street. The same idea works well in Python when we compute the x/y coordinates directly.

In [ ]:
coords = np.column_stack((deaths_proj.geometry.x.to_numpy(), deaths_proj.geometry.y.to_numpy()))
weights = deaths_proj['Num_Cases'].to_numpy()

mean_center = coords.mean(axis=0)
weighted_center = np.average(coords, axis=0, weights=weights)

# Standard distance (weighted): average radius around the weighted center
distances = np.sqrt(((coords - weighted_center) ** 2).sum(axis=1))
standard_distance = np.sqrt(np.average(distances ** 2, weights=weights))

# Weighted covariance and ellipse axes
cov = np.cov(coords.T, aweights=weights)
eigvals, eigvecs = np.linalg.eigh(cov)
order = eigvals.argsort()[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]
angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
ellipse_width = 2 * np.sqrt(eigvals[0])
ellipse_height = 2 * np.sqrt(eigvals[1])

print('Mean center:', mean_center)
print('Weighted center:', weighted_center)
print('Standard distance:', standard_distance)
print('Ellipse width/height:', ellipse_width, ellipse_height)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
deaths_proj.plot(ax=ax, color='crimson', markersize=np.clip(deaths_proj['Num_Cases'] * 12, 12, 120), alpha=0.5)

ax.scatter(mean_center[0], mean_center[1], marker='x', s=150, color='black', label='Mean center')
ax.scatter(weighted_center[0], weighted_center[1], marker='*', s=220, color='gold', edgecolor='black', label='Weighted mean center')

ellipse = Ellipse(
    xy=weighted_center,
    width=ellipse_width * 2.5,
    height=ellipse_height * 2.5,
    angle=angle,
    fill=False,
    edgecolor='navy',
    linewidth=2,
    label='Standard ellipse (illustrative)'
)
ax.add_patch(ellipse)

ax.set_title('Mean center, weighted mean center, and ellipse')
ax.set_axis_off()
ax.legend(loc='lower left')
plt.show()

## Kernel density hotspot surface

Kernel density turns the death points into a smooth surface, which is a helpful way to see the hotspot around Broad Street.

In [ ]:
# Build a grid over the study area and evaluate a weighted Gaussian kernel density surface.
xmin, ymin, xmax, ymax = study_proj.total_bounds
xgrid, ygrid = np.mgrid[xmin:xmax:250j, ymin:ymax:250j]
positions = np.vstack([xgrid.ravel(), ygrid.ravel()])

kde = gaussian_kde(coords.T, weights=weights, bw_method=0.15)
density = np.reshape(kde(positions).T, xgrid.shape)

fig, ax = plt.subplots(figsize=(10, 10))
img = ax.imshow(
    np.rot90(density),
    cmap='magma',
    extent=[xmin, xmax, ymin, ymax],
    aspect='equal'
)
study_proj.boundary.plot(ax=ax, color='white', linewidth=1.5)
pumps_proj.plot(ax=ax, color='cyan', markersize=30)
plt.colorbar(img, ax=ax, shrink=0.75, label='Kernel density')
ax.set_title('Kernel density surface for the John Snow deaths')
ax.set_axis_off()
plt.show()

## Optional network-based service areas

The old bullet list mentioned service areas that behave like Voronoi polygons, but use **travel time** instead of straight-line distance. That is a perfect extension for a notebook because you can use `osmnx` and `networkx` to build a walk network and compute isochrones or shortest-path catchments.

A sketch of the workflow looks like this:

1. Download a walking network around Soho / Broad Street.
2. Add the water pump locations to the graph.
3. Compute shortest-path travel time or distance to each node.
4. Assign each node to the nearest pump in network space.
5. Polygonize the resulting service areas.

```python
# Optional extension only
# import osmnx as ox
# G = ox.graph_from_place('Soho, London, UK', network_type='walk')
# G = ox.project_graph(G, to_crs=analysis_crs)
# ... continue with travel-time service area analysis ...
```

> If you do not have time for the network extension, the Voronoi polygons and the spatial join already capture the main historical story very well.

## Optional: display the historic map image

If you installed `rasterio`, you can also view the historic map from the archive as a background layer.

In [ ]:
try:
    import rasterio
    from rasterio.plot import show

    fig, ax = plt.subplots(figsize=(10, 10))
    with rasterio.open(MAP_PATH) as src:
        show(src, ax=ax)
    ax.set_title('Historic John Snow map from the archive')
    plt.show()
except Exception as exc:
    print('Raster display skipped:', exc)

## Turn-in / notebook checklist

If you are using this as a lab submission, make sure your notebook includes:

- a Voronoi service-area map
- a spatial join table showing deaths allocated to pumps
- a summary of deaths by pump name
- a mean-center / weighted-mean-center plot
- a standard distance or ellipse visualization
- a kernel density hotspot surface

### Files used in this lab

- `Being_John_Snow.zip` (downloaded automatically in Colab)
- `Being_John_Snow/data/raw/Water_Pumps.geojson`
- `Being_John_Snow/data/raw/deathAddresses.csv`
- `Being_John_Snow/data/raw/Study_Area.shp`
- `Being_John_Snow/data/raw/John_Snow_Map.tif`

The notebook file itself is stored in the course repository at `data/Being_John_Snow_Python_Notebook.ipynb`.